# Andes Capital Real Estate
## 02. Data Preparation & Transformation

### Objective

Prepare and validate the sales, customer, and property datasets for business analysis and Power BI reporting.

### Approach

- Preserve the original datasets.
- Apply only necessary transformations.
- Validate business rules and financial consistency.
- Document all data preparation decisions.

In [1]:
import pandas as pd
from pathlib import Path

# Define project directories
project_dir = Path.cwd()

if not (project_dir / "data").exists():
    project_dir = project_dir.parent

raw_dir = project_dir / "data" / "raw"
processed_dir = project_dir / "data" / "processed"

# Load original datasets
sales = pd.read_csv(raw_dir / "hecho_ventas_propiedades.csv")
customers = pd.read_csv(raw_dir / "dim_clientes.csv")
properties = pd.read_csv(raw_dir / "dim_propiedades.csv")

print("Datasets loaded successfully!")

Datasets loaded successfully!


In [2]:
# Compare sale prices with listing prices

price_check = sales.merge(
    properties[["id_propiedad", "precio_publicado"]],
    on="id_propiedad",
    how="left",
    validate="many_to_one"
)

display(price_check[
    ["id_venta", "precio_venta", "precio_publicado"]
].head())


,id_venta,precio_venta,precio_publicado
0,SALE000001,1027126,1075248
1,SALE000002,233363,248261
2,SALE000003,209704,238246
3,SALE000004,224411,229061
4,SALE000005,192007,208029


In [3]:
# Calculate price differences

price_check["diferencia_precio"] = (
    price_check["precio_venta"] - price_check["precio_publicado"]
)

price_check["diferencia_pct"] = (
    price_check["diferencia_precio"]
    / price_check["precio_publicado"]
) * 100

display(
    price_check[
        ["precio_venta", "precio_publicado",
         "diferencia_precio", "diferencia_pct"]
    ].describe().round(2)
)

,precio_venta,precio_publicado,diferencia_precio,diferencia_pct
count,8500.00,8500.00,8500.00,8500.00
mean,707353.20,760764.28,-53411.09,-7.06
std,645833.19,692681.39,56330.43,2.89
min,76837.00,85548.00,-563112.00,-12.00
25%,307004.25,330339.00,-65014.50,-9.61
50%,489123.50,527287.50,-34935.00,-7.04
75%,866908.50,933127.75,-19623.25,-4.58
max,4944525.00,5206092.00,-2238.00,-2.00


### Listing Price vs. Sale Price Validation

All 8,500 transactions have a recorded sale price below the corresponding listing price.

**Key findings:**
- Average price difference: -7.06%
- Median price difference: -7.04%
- Observed range: -12.00% to -2.00%
- No transactions exceeded their recorded listing price.

**Interpretation:**

The results indicate a consistent gap between listing and final sale prices.

However, the dataset does not establish whether listing prices correspond to the prices in effect at the time of each transaction.

**Decision:**

Preserve the original prices. The percentage difference may be considered for exploratory commercial analysis, but it should not be presented as a confirmed negotiation discount.

In [4]:
# Validate property attributes

print("Properties with negative bedrooms:",
      (properties["habitaciones"] < 0).sum())

print("Properties with zero or negative area:",
      (properties["tamano_m2"] <= 0).sum())

print("Properties with zero or negative listing price:",
      (properties["precio_publicado"] <= 0).sum())

Properties with negative bedrooms: 0
Properties with zero or negative area: 0
Properties with zero or negative listing price: 0


In [5]:
# Check bedroom distribution by property type

bedroom_distribution = pd.crosstab(
    properties["tipo_propiedad"],
    properties["habitaciones"]
)

display(bedroom_distribution)

habitaciones,0,1,2,3,4,5,6
tipo_propiedad,,,,,,,
Casa,0,0,225,745,820,318,109
Comercial,977,0,0,0,0,0,0
Departamento,0,992,2151,1440,223,0,0


In [6]:
# Create working copies

sales_clean = sales.copy()
customers_clean = customers.copy()
properties_clean = properties.copy()

In [7]:
# Convert sales dates to datetime

sales_clean["fecha_venta"] = pd.to_datetime(
    sales_clean["fecha_venta"],
    format="%Y-%m-%d"
)

print(sales_clean["fecha_venta"].dtype)

datetime64[us]


In [8]:
# Select columns for the fact table

sales_clean = sales_clean[
    [
        "id_venta",
        "fecha_venta",
        "id_cliente",
        "id_propiedad",
        "precio_venta",
        "canal_venta",
        "porcentaje_comision",
        "monto_comision"
    ]
].copy()

display(sales_clean.head())

,id_venta,fecha_venta,id_cliente,id_propiedad,precio_venta,canal_venta,porcentaje_comision,monto_comision
0,SALE000001,2024-01-05,CUST02497,PROP03591,1027126,Corredor,0.0473,48605
1,SALE000002,2024-06-16,CUST02903,PROP03005,233363,Corredor,0.0458,10681
2,SALE000003,2024-10-13,CUST00831,PROP04661,209704,Directo,0.0151,3156
3,SALE000004,2023-02-08,CUST01686,PROP06300,224411,Corredor,0.0380,8523
4,SALE000005,2024-11-29,CUST00639,PROP00792,192007,Directo,0.0159,3054


In [9]:
# Validate the prepared fact table

print(f"Original records: {len(sales):,}")
print(f"Prepared records: {len(sales_clean):,}")

print(f"\nOriginal sales total: {sales['precio_venta'].sum():,.2f}")
print(f"Prepared sales total: {sales_clean['precio_venta'].sum():,.2f}")

print(f"\nOriginal commissions: {sales['monto_comision'].sum():,.2f}")
print(f"Prepared commissions: {sales_clean['monto_comision'].sum():,.2f}")

Original records: 8,500
Prepared records: 8,500

Original sales total: 6,012,502,170.00
Prepared sales total: 6,012,502,170.00

Original commissions: 200,627,166.00
Prepared commissions: 200,627,166.00


In [10]:
# Validate dimension tables

for name, original, prepared in [
    ("Customers", customers, customers_clean),
    ("Properties", properties, properties_clean)
]:
    print(f"\n{name}")
    print(f"Original shape: {original.shape}")
    print(f"Prepared shape: {prepared.shape}")
    print(f"Data unchanged: {original.equals(prepared)}")


Customers
Original shape: (3500, 4)
Prepared shape: (3500, 4)
Data unchanged: True

Properties
Original shape: (8000, 8)
Prepared shape: (8000, 8)
Data unchanged: True


In [11]:
# Export prepared datasets

processed_dir.mkdir(parents=True, exist_ok=True)

sales_clean.to_csv(
    processed_dir / "fact_sales.csv",
    index=False,
    date_format="%Y-%m-%d"
)

customers_clean.to_csv(
    processed_dir / "dim_customers.csv",
    index=False
)

properties_clean.to_csv(
    processed_dir / "dim_properties.csv",
    index=False
)

print("Prepared datasets exported successfully!")

Prepared datasets exported successfully!
